# 02_Data_Cleaning — Data Quality & Validation

In [1]:
import os
if 'notebooks' in os.getcwd():
    os.chdir('..')
print(f'Working directory: {os.getcwd()}')

Working directory: D:\Data analyst\Portfolio\project-1-refugee-integration


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
print('✓ Libraries loaded')

✓ Libraries loaded


In [3]:
master = pd.read_parquet('data/processed/refugee_master_consolidated.parquet')
print(f'Shape: {master.shape}')
print(f'Memory: {master.memory_usage(deep=True).sum() / (1024*1024):.2f} MB')

Shape: (29506, 27)
Memory: 8.13 MB


## 1. Null Values Analysis

In [4]:
nulls = master.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print('Nulls by column:')
for col, count in nulls.items():
    pct = count / len(master) * 100
    print(f'  {col}: {count} ({pct:.2f}%)')

if len(nulls) == 0:
    print('  ✓ No nulls found')

Nulls by column:
  ✓ No nulls found


## 2. Outlier Detection

In [5]:
components = ['employment_rate', 'housing_access', 'education_enrollment', 'income_stability', 'legal_rights_score', 'social_cohesion']

print('Outliers (IQR method):')
for col in components:
    Q1 = master[col].quantile(0.25)
    Q3 = master[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = master[(master[col] < Q1 - 1.5 * IQR) | (master[col] > Q3 + 1.5 * IQR)]
    print(f'  {col}: {len(outliers)} ({len(outliers)/len(master)*100:.2f}%)')

Outliers (IQR method):
  employment_rate: 120 (0.41%)
  housing_access: 175 (0.59%)
  education_enrollment: 388 (1.31%)
  income_stability: 0 (0.00%)
  legal_rights_score: 12 (0.04%)
  social_cohesion: 176 (0.60%)


## 3. Integration Index Validation

In [6]:
print('Integration Index Stats:')
print(master['integration_index'].describe().round(2))

out_of_bounds = master[(master['integration_index'] < 0) | (master['integration_index'] > 100)]
print(f'\nOut of bounds (0-100): {len(out_of_bounds)}')
print(f'NaN values: {master["integration_index"].isna().sum()}')

Integration Index Stats:
count    29506.00
mean        54.12
std          8.64
min         28.40
25%         48.30
50%         53.74
75%         59.92
max         79.50
Name: integration_index, dtype: float64

Out of bounds (0-100): 0
NaN values: 0


## 4. Completeness Summary

In [7]:
completeness = (1 - master.isnull().sum() / len(master)) * 100
print('Data Completeness:')
for col, pct in completeness[completeness < 100].sort_values().items():
    print(f'  {col}: {pct:.1f}%')

print(f'\nOverall: {completeness.mean():.1f}%')

Data Completeness:

Overall: 100.0%


## 5. Final Checklist

In [8]:
checks = {
    'No nulls (core)': master[['iso_a3', 'Year', 'integration_index']].isnull().sum().sum() == 0,
    'Index in range': ((master['integration_index'] >= 0) & (master['integration_index'] <= 100)).all(),
    'All countries mapped': master['iso_a3'].notna().all(),
    'Years 2000-2023': ((master['Year'] >= 2000) & (master['Year'] <= 2023)).all(),
    'No duplicates': master.duplicated().sum() == 0
}

print('Quality Checklist:')
for check, result in checks.items():
    print(f'  {"✓" if result else "✗"} {check}')

print(f'\n{"✓ ALL PASSED" if all(checks.values()) else "✗ ISSUES FOUND"}')

Quality Checklist:
  ✓ No nulls (core)
  ✓ Index in range
  ✓ All countries mapped
  ✓ Years 2000-2023
  ✓ No duplicates

✓ ALL PASSED
